# Arnav's model formulation and scaling

Updated 28 October 2026 for the completed historical iteration at [28dbe1e](https://github.com/FriedelCraft/lynx-hare-project/tree/28dbe1e2a611719f3d65e5922db528ae423724f2). This notebook explains the mathematical foundation and checks coordinate changes. It does not fit observations, train a PINN, or score forecasts.

Use **Restart Kernel and Run All** with Python 3; no third-party package is needed. All numerical examples below are synthetic algebra inputs, not historical parameter estimates. The main scaling matches Joel's x=H/K, y=aP/(rK), s=rt. The report gives the derivations; finite exact-arithmetic checks do not replace symbolic proof or empirical validation.

The assistant executed the code cells in order in a fresh namespace and saved their outputs. Arnav's own review is recorded separately.


## Phase 1 Question, observations and assumptions

The completed historical iteration uses the frozen Stan transcription of annual pelt counts in thousands, not animal censuses ([Carpenter 2018](https://mc-stan.org/learn-stan/case-studies/lotka-volterra-predator-prey.html); [Advay's data notes](https://github.com/FriedelCraft/lynx-hare-project/blob/28dbe1e2a611719f3d65e5922db528ae423724f2/data/historical/README.md)). H and P denote noiseless pelt indices, with observations

$$Y_{H,i}=H(t_i)+\varepsilon_{H,i},\quad Y_{P,i}=P(t_i)+\varepsilon_{P,i},\quad t_i=\mathrm{Year}_i-1900.$$

Under O1, $H=c_H N_H$ and $P=c_P N_P$ for constant unknown positive factors and latent animal abundances N. We fit no observation-scale factors: the effective coefficients absorb them. This does not assume that those factors equal one. Annual totals are approximated as point-state observations. H and P are in thousands of pelts, and their derivatives in thousands of pelts per year.

The [reference configuration](https://github.com/FriedelCraft/lynx-hare-project/blob/28dbe1e2a611719f3d65e5922db528ae423724f2/configs/historical.json) fits all six positive coefficients without an e<=1 or coexistence constraint. It uses model.solve, RK45, rtol=1e-8, atol=1e-10, max_step=0.1. The historical training maxima are (77.4,59.4); the different rational normalization values in the algebra cells below are deliberately synthetic test inputs.

The main ODE begins at (30,4) in 1900. The PINN uses a soft initial penalty and reports its discrepancy. The separate sensitivity fits initial states within 0.5 to 1.5 times observed; another uses log residuals. Fitting and scale selection use only 1900-1915. Frozen forecasts start at 1900, with no test-year restart, and separate species MAE/RMSE score 1916-1920. This notebook reads no data tables and does not repeat scoring. Source: [Yash's historical handoff](https://github.com/FriedelCraft/lynx-hare-project/blob/28dbe1e2a611719f3d65e5922db528ae423724f2/docs/yash_historical_handoff.md).

Assume a closed, spatially mixed two-species system, continuous states, constant parameters, logistic prey limitation and predator dependence on prey. Alternative prey, explicit harvesting and seasonal forcing are omitted. K,b,a,e are effective pelt-unit coefficients. r,m and the dimensionless groups are scale invariant, but their biological interpretation remains conditional on O1 and model adequacy. Stable-cycle classifications refer to the fitted model, not an established property of the real ecosystem.


## Phase 2 Why this model

Lotka–Volterra has $\dot H=rH-cHP$ and $\dot P=ecHP-mP$. Its positive equilibrium
is $H_*=m/(ec), P_*=r/c$. For positive states define

$$V=ec[H-H_*-H_*\log(H/H_*)]+c[P-P_*-P_*\log(P/P_*)].$$

Differentiating gives
$$\dot V=(ecH-m)(r-cP)+(cP-r)(ecH-m)=0.$$
The closed level curves imply neutral periodic orbits rather than attraction to
one selected cycle. The prey also grows exponentially without predators.

Use logistic growth $rH(1-H/K)$ and a saturating consumption rate per predator
$g(H)=aH/(b+H)$. A search-and-handling time budget gives
$g/(qH)+g/a=1$, hence $g=aqH/(a+qH)$ with $b=a/q$.
Thus $g(b)=a/2$, $g(H)\to a$ as $H\to\infty$, and $g(H)\approx(a/b)H$ at low prey.
This is the type II response attributed to
[Holling (1959)](https://doi.org/10.4039/Ent91385-7).

The chosen [Rosenzweig–MacArthur model](https://doi.org/10.1086/282272) is
$$f_H=rH(1-H/K)-aHP/(b+H),\quad f_P=eaHP/(b+H)-mP.$$

| Quantity | Units |
|---|---|
| $r,m$ | year$^{-1}$ |
| $K,b,H$ | $U_H$ |
| $P$ | $U_P$ |
| $a$ | $U_H/(U_P\,\mathrm{year})$ |
| $e$ | $U_P/U_H$ |

All parameters are positive. Index conversion $e$ is not automatically a biomass
efficiency in $[0,1]$. Each derivative has its own state as a factor, so exact
nonnegative solutions stay nonnegative. Also $\dot H\le rH(1-H/K)$.
These are consistency properties, not empirical validation.

## Phase 3 Non-dimensionalisation and the checking code

To match Joel, use $s=rt$, $x=H/K$, $y=aP/(rK)$ and
$\beta=b/K$, $\alpha=ea/r$, $\delta=m/r$.
The chain rule gives $\dot H=rKx_s$ and $\dot P=(r^2K/a)y_s$.
Substitution and division therefore yield

$$x_s=x(1-x)-\frac{xy}{\beta+x},\qquad
y_s=\frac{\alpha xy}{\beta+x}-\delta y.$$

Return to original units with $H=Kx$, $P=(rK/a)y$, and Year $=1900+s/r$.
If $r,K,a$ are specified separately, $b=\beta K$, $e=\alpha r/a$, $m=\delta r$.
The three groups alone do not identify six dimensional parameters.

**Decision:** match the existing team notation. The alternative $H/b$, $aP/(rb)$
keeps state scales fixed in a $K$ sweep, but introduces a second convention.
With our choice, recompute scaled initial conditions whenever $K$ changes.

The next cell defines the two right-hand sides and the group map directly from
these equations. `Fraction(n,d)` represents exactly $n/d$
([Python documentation](https://docs.python.org/3/library/fractions.html)).
`equal` records an equality and raises an error if it fails. `product` enumerates
the state pairs. No random seeds or data files are required.

The first parameter set is [Yash's synthetic configuration](https://github.com/FriedelCraft/lynx-hare-project/blob/7d17deee1cb685f016ff16cb6706e8db12ebe6b2/configs/synthetic.json).
The second is Joel's illustrative set with $K=60$; the third is an arbitrary
positive arithmetic example. These deliberately include $r\ne1$.

In [1]:
from fractions import Fraction as F
from itertools import product

checks = {}

def equal(group, actual, expected):
    if actual != expected:
        raise AssertionError(f"{group}: {actual} != {expected}")
    checks[group] = checks.get(group, 0) + 1

def dimensional_rhs(H, P, p):
    consumption = p["a"] * H * P / (p["b"] + H)
    return (p["r"] * H * (1 - H / p["K"]) - consumption,
            p["e"] * consumption - p["m"] * P)

def groups(p):
    return p["b"] / p["K"], p["e"] * p["a"] / p["r"], p["m"] / p["r"]

def scaled_rhs(x, y, beta, alpha, delta):
    consumption = x * y / (beta + x)
    return x * (1 - x) - consumption, alpha * consumption - delta * y

names = ("r", "K", "a", "b", "e", "m")
parameter_sets = [dict(zip(names, values)) for values in (
    (F(1), F(60), F(2), F(20), F(3, 5), F(1, 2)),
    (F(3, 2), F(60), F(4), F(30), F(2, 5), F(4, 5)),
    (F(7, 10), F(80), F(5, 2), F(15), F(4, 5), F(3, 10)),
)]
states = list(product((F(0), F(7, 3), F(35), F(90)),
                      (F(0), F(9), F(25, 2))))
print(f"Prepared {len(parameter_sets)} parameter sets and {len(states)} states.")

Prepared 3 parameter sets and 12 states.


### Check 1 Derivatives and reverse transformations

For each state, evaluate the dimensional equations independently, transform the
state, evaluate the scaled equations, and convert their derivatives back.
Both results must agree exactly. The boundary states and $H>K$ cases check that
the transformation works beyond a single positive example.

The code also checks state/time round trips and reconstructs $b,e,m$ using the
groups plus the specified scales. The final equalities verify the numerical
fractions printed in the report.

In [2]:
for p in parameter_sets:
    r, K, a = p["r"], p["K"], p["a"]
    beta, alpha, delta = groups(p)
    equal("parameter_round_trip", (beta*K, alpha*r/a, delta*r),
          (p["b"], p["e"], p["m"]))
    t = F(7, 3)
    equal("time_round_trip", (r*t)/r, t)
    for H, P in states:
        x, y = H/K, a*P/(r*K)
        dx, dy = scaled_rhs(x, y, beta, alpha, delta)
        equal("transformed_derivatives", (r*K*dx, r*r*K/a*dy),
              dimensional_rhs(H, P, p))
        equal("state_round_trip", (K*x, r*K/a*y), (H, P))

p = parameter_sets[0]
equal("report_example", groups(p), (F(1, 3), F(6, 5), F(1, 2)))
equal("report_example", dimensional_rhs(F(35), F(9), p), (F(413, 132), F(261, 110)))
equal("report_example", scaled_rhs(F(7, 12), F(3, 10), *groups(p)),
      (F(413, 7920), F(87, 1100)))
print("Derivative conversions and reverse transformations passed for 36 cases.")
print("The three report example equalities also passed.")

Derivative conversions and reverse transformations passed for 36 cases.
The three report example equalities also passed.


## Phase 4 Compatibility with numerical normalization

[Yash's method](https://github.com/FriedelCraft/lynx-hare-project/blob/7d17deee1cb685f016ff16cb6706e8db12ebe6b2/docs/method.md)
uses $u=H/s_H$, $v=P/s_P$ with training-only maxima and internal time
$\xi=2t/T-1$. In elapsed years,

$$u_t=ru(1-s_Hu/K)-\frac{a s_Puv}{b+s_Hu},\quad
v_t=\frac{ea s_Huv}{b+s_Hu}-mv.$$

Our mapping is $x=(s_H/K)u$, $y=(a s_P/(rK))v$ and $s=rT(\xi+1)/2$.
Since $r,a$ are inferred in the current experiment, keep the fixed numerical
normalization rather than changing training coordinates as the parameters move.

**Check 2:** the two normalized right-hand sides must equal $f_H/s_H,f_P/s_P$.
With arbitrary nonzero network slopes, verify
$R_t=(2/T)u_\xi-f_H/s_H$ and $R_\xi=(T/2)R_t$ (likewise for $v$).
The squared penalty therefore changes by $(T/2)^2$; compensate with weight
$(2/T)^2$ to preserve the objective. The illustrative $s_H,s_P$ below are arbitrary
positive scales used only to check algebra; they are not claimed training maxima.

In [3]:
sH, sP = F(71, 2), F(43, 3)
for p in parameter_sets:
    r, K, a, b, e, m = (p[name] for name in names)
    for H, P in states:
        u, v = H/sH, P/sP
        fH, fP = dimensional_rhs(H, P, p)
        normalized = (r*u*(1-sH*u/K) - a*sP*u*v/(b+sH*u),
                      e*a*sH*u*v/(b+sH*u) - m*v)
        equal("normalized_rhs", normalized, (fH/sH, fP/sP))
        equal("coordinate_bridge", (sH/K*u, a*sP/(r*K)*v), (H/K, a*P/(r*K)))
        for T in (F(15), F(7, 2)):
            u_xi, v_xi = F(5, 7), F(-2, 9)  # Arbitrary slopes, not fitted outputs.
            Rt = (2/T*u_xi-fH/sH, 2/T*v_xi-fP/sP)
            Rxi = (u_xi-T/2*fH/sH, v_xi-T/2*fP/sP)
            equal("time_chain_rule", Rxi, tuple(T/2*z for z in Rt))
            equal("loss_weight_conversion",
                  F(10)*(2/T)**2*sum(z*z for z in Rxi)/2,
                  F(10)*sum(z*z for z in Rt)/2)
print("Normalized equations passed for 36 cases; chain rule and loss weights for 72.")

Normalized equations passed for 36 cases; chain rule and loss weights for 72.


### Check 3 Changes of measurement units

For $\widetilde H=c_HH$, $\widetilde P=c_PP$, keep $r,m$ and transform
$K,b$ by $c_H$, $a$ by $c_H/c_P$ and $e$ by $c_P/c_H$.
Substitution into each dimensional equation gives
$\widetilde f_H=c_Hf_H$, $\widetilde f_P=c_Pf_P$.
The dimensionless groups and states remain invariant.

This is a derivation about constant scales. If pelt-to-population factors are
unknown, it explains why absolute biological rates cannot be recovered just by
renaming pelt counts as populations. It does not prove non-identifiability of all
effective coefficients when the measurement scale is fixed and known.

In [4]:
for p in parameter_sets:
    for cH, cP in ((F(1, 1000), F(1, 1000)), (F(2), F(3)), (F(3, 7), F(5, 2))):
        transformed = dict(p, K=cH*p["K"], b=cH*p["b"],
                           a=cH/cP*p["a"], e=cP/cH*p["e"])
        equal("group_unit_invariance", groups(transformed), groups(p))
        for H, P in states:
            fH, fP = dimensional_rhs(H, P, p)
            equal("unit_change_rhs", dimensional_rhs(cH*H, cP*P, transformed),
                  (cH*fH, cP*fP))
            equal("state_unit_invariance",
                  (cH*H/transformed["K"], transformed["a"]*cP*P/(p["r"]*transformed["K"])),
                  (H/p["K"], p["a"]*P/(p["r"]*p["K"])))
print("Measurement-unit changes passed for 108 state/scale cases.")

Measurement-unit changes passed for 108 state/scale cases.


## Phase 5 Boundary behaviour and the Lotka–Volterra reference

**Check 4:** zero prey has zero prey derivative; zero predators have zero predator
derivative. With $H=0$, predators decline as $-mP$; with $P=0$, prey follows the
logistic equation. These are exact boundary checks, not numerical integrations.

For positive states, check the derivative of the Lotka–Volterra conserved
quantity. Here the reference encounter coefficient is set to $c=a/b$ purely for
comparison. Low-prey saturation alone does not remove logistic growth: the
classical reference also needs $H/K\ll1$. For the response alone, the linear
approximation overestimates $g(H)$ by the relative amount $H/b$, measured relative
to the exact response. The notebook checks that identity too.

In [5]:
for p in parameter_sets:
    r, K, a, b, e, m = (p[name] for name in names)
    equal("boundary_limits", dimensional_rhs(F(0), F(9), p), (F(0), -m*9))
    equal("boundary_limits", dimensional_rhs(F(35), F(0), p),
          (r*35*(1-F(35)/K), F(0)))
    c = a/b
    for H, P in states:
        if H > 0 and P > 0:
            dH = r*H-c*H*P
            dP = e*c*H*P-m*P
            dV = (e*c-m/H)*dH + (c-r/P)*dP
            equal("lv_conserved_derivative", dV, F(0))
    for ratio in (F(1, 100), F(1, 10), F(1)):
        H = b*ratio
        exact_response = a*H/(b+H)
        linear_response = a/b*H
        equal("low_prey_response_error", (linear_response-exact_response)/exact_response, ratio)
    equal("half_saturation", a*b/(b+b), a/2)
print("Boundary limits, the conserved derivative and response identities passed.")

Boundary limits, the conserved derivative and response identities passed.


## Phase 6 Results and interpretation

The final cell prints the number of exact equality assertions in each group.
The group counts include related checks; they are not independent scientific
experiments. The quantitative discrepancy is zero at every tested rational input.

This verifies the algebra in the implementation of the transformations. It does
not measure forecasting skill, establish global identifiability, or validate the
model against ecology. Joel's analysis and Rishi's numerical checks address other
mathematical questions. Advay and Yash must compare forecasts against held-out
historical observations using separately reported species MAE/RMSE and graphs.

Before presenting this notebook, rederive the chain rule and explain each function
and loop. Record your own review in `ai_acknowledgement.md`; the saved outputs do
not establish that personal review has occurred.

In [6]:
for name, count in checks.items():
    print(f"{name}: {count} exact equality checks passed")
print(f"Total: {sum(checks.values())} exact equalities; discrepancy = 0 at tested inputs.")

parameter_round_trip: 3 exact equality checks passed
time_round_trip: 3 exact equality checks passed
transformed_derivatives: 36 exact equality checks passed
state_round_trip: 36 exact equality checks passed
report_example: 3 exact equality checks passed
normalized_rhs: 36 exact equality checks passed
coordinate_bridge: 36 exact equality checks passed
time_chain_rule: 72 exact equality checks passed
loss_weight_conversion: 72 exact equality checks passed
group_unit_invariance: 9 exact equality checks passed
unit_change_rhs: 108 exact equality checks passed
state_unit_invariance: 108 exact equality checks passed
boundary_limits: 6 exact equality checks passed
lv_conserved_derivative: 18 exact equality checks passed
low_prey_response_error: 9 exact equality checks passed
half_saturation: 3 exact equality checks passed
Total: 558 exact equalities; discrepancy = 0 at tested inputs.


## References and attribution

1. Rosenzweig, M. L. and MacArthur, R. H. (1963), *The American Naturalist* 97(895), 209–223. [DOI](https://doi.org/10.1086/282272).
2. Holling, C. S. (1959), *The Canadian Entomologist* 91(7), 385–398. [DOI](https://doi.org/10.4039/Ent91385-7).
3. Carpenter, B. (2018), *Predator-Prey Population Dynamics: the Lotka-Volterra Model in Stan*. [Case study](https://mc-stan.org/learn-stan/case-studies/lotka-volterra-predator-prey.html).
4. Lynx Hare Project team, revision `7d17deee1cb685f016ff16cb6706e8db12ebe6b2`, accessed 26 October 2026. [Repository](https://github.com/FriedelCraft/lynx-hare-project/tree/7d17deee1cb685f016ff16cb6706e8db12ebe6b2). Specific model, configuration, Yash method and Joel theory files are linked where used.
5. Python Software Foundation, *fractions — Rational numbers*, accessed 26 October 2026. [Documentation](https://docs.python.org/3/library/fractions.html).
6. The supplied MAC F314 work-distribution document and professor's instructions in chat define the scope and evaluation requirements; they are not external scientific evidence.

AI assistance: OpenAI Codex helped draft the exposition and checking code. General
derivations are written out in the report. No unlisted book or source is claimed
to have been consulted. The optional longer tsibbledata series is not used here.